In [ ]:
import pandas as pd 
import os
import matplotlib.pyplot as plt
import matplotlib.patches
from matplotlib import cm
from IPython.display import Image, display
import numpy as np
import sys
from scipy.stats import linregress
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from math import sqrt
import math
from functools import reduce
from matplotlib import colors
from matplotlib.colors import ListedColormap
from matplotlib.ticker import PercentFormatter
from matplotlib.lines import Line2D
import shap
import time
from datetime import date
from datetime import datetime

In [ ]:
def set_params(feature_name, ttrs):
    tree_dir_dict = {
        'old-pol_old-pi_y-ecfp_all-torsion_all-sd': [150, 'ooyaa'],
        'old-pol_old-pi_y-ecfp_z-torsion_x-sd': [175, 'ooyzx'],
        'new-pol_old-pi_y-ecfp_z-torsion_x-sd': [100, 'noyzx'],
        'new-pol_new-pi_y-ecfp_z-torsion_x-sd': [125, 'nnyzx'],
        'old-pol_old-pi_y-ecfp_z-torsion_x-sd_pt': [150, 'ooyzx_pt'],
    }
    
    feature_name_list = feature_name.split('-')
    if 'pt' in feature_name_list:
        feature_code = feature_name_list[0] + '-pol_' + feature_name_list[1] + '-pi_' + feature_name_list[2] + '-ecfp_' + feature_name_list[3] + '-torsion_' + feature_name_list[4] + '-sd' + '_pt'
    else:
        feature_code = feature_name_list[0] + '-pol_' + feature_name_list[1] + '-pi_' + feature_name_list[2] + '-ecfp_' + feature_name_list[3] + '-torsion_' + feature_name_list[4] + '-sd'
    
    num_trees = tree_dir_dict.get(feature_code)[0]
    desc_str = tree_dir_dict.get(feature_code)[1]
    output_path = os.path.join(os.getcwd(), 'output', desc_str)
    os.makedirs(output_path, exist_ok=True)
    
    _, m, d = str(date.today()).split('-')
    mddate = m + d
    
    suffix = f"{desc_str}-{ttrs}-{mddate}"
    
    params = [feature_code, suffix, output_path]
    return num_trees, params


In [ ]:
def process_df(df, feature_code, df_type):
    # feature_code: [old/new/no]-pol_[old/new/no]-pi_[y/n]-ecfp_[all/no/x/y/z]-torsion_[all/no/x/y/z]-sd
    if feature_code.endswith('_pt'):
        feature_code = feature_code.split('_pt')[0]
        first_cols = ['Monomer 1', 'Monomer 2']
    else:
        first_cols = ['Monomer 1', 'Monomer 2', 'Oligomer Type']
        
    pol, pi, ecfp = feature_code.split('_')[:-2]
    torsion_selection = feature_code.split('_')[-2]
    sd_selection = feature_code.split('_')[-1]
    
    geo_options = ['x', 'y', 'z']
    torsion_options = ['Total Torsion Angle', 'Mean Torsion Angle', 'Inner Mean Torsion Angle']
    sd_options = ['SD Torsions', 'SD Difference']
    
    geo_select_dict = {
        'all-torsion': [f"{t_opt}_{g_opt}" for t_opt in torsion_options for g_opt in geo_options],
        'all-sd': [f"{s_opt}_{g_opt}" for s_opt in sd_options for g_opt in geo_options],
        'no-torsion': [],
        'no-sd': [],
        'x-torsion': [f"{t_opt}_x" for t_opt in torsion_options],
        'y-torsion': [f"{t_opt}_y" for t_opt in torsion_options],
        'z-torsion': [f"{t_opt}_z" for t_opt in torsion_options],
        'x-sd': [f"{s_opt}_x" for s_opt in sd_options],
        'y-sd': [f"{s_opt}_y" for s_opt in sd_options],
        'z-sd': [f"{s_opt}_z" for s_opt in sd_options],
    }
    
    others_select_dict = {
        'old-pol': ['Molecule Polarizability', 'Average Monomer Polarizability', 'Dipole Moment'],
        'new-pol': ['Rotated Polarizability', 'Average Monomer Polarizability', 'Dipole Moment', 'α_xx', 'α_xy', 'α_yy'],
        'old-pi': ['PiSystemSize'],
        'new-pi': ['PiSystemSize', 'PiSystemArea', 'PiSystemLength'],
        'y-ecfp': [f"ECFP_{i}" for i in range(4096)],
        'no-pi': [],
        'no-pol': [],
        'n-ecfp': []
    }
    
    pol_cols = others_select_dict.get(pol)
    torsion_cols = geo_select_dict.get(torsion_selection)
    sd_cols = geo_select_dict.get(sd_selection)
    pi_cols = others_select_dict.get(pi)
    ecfp_cols = others_select_dict.get(ecfp)
    spec_cols = ['DFT Oscillator Strength', 'DFT CD Intensity']
    
    if pol.split('-')[0] != 'no' and pi.split('-')[0] != 'no':
        energy_cols = ['Molecule HOMO (eV)', 'Molecule LUMO (eV)', 'Molecule HLG (eV)', 'ΔE_HOMO (eV)', 'ΔE_LUMO (eV)', 'Monomer HLG (eV)']
        first_cols.extend(energy_cols)

    col_order = first_cols + pol_cols + torsion_cols + sd_cols + pi_cols + ecfp_cols + spec_cols
    gfn_drop = ['DFT Oscillator Strength', 'DFT CD Intensity']
    col_order = [c for c in col_order if c not in gfn_drop] if df_type == 'gfn' else col_order
    df = df[col_order]
    
    df = df.rename(columns = {
        'DFT CD Intensity': 'DFT CD Intensity (a.u.)',
        'DFT Oscillator Strength': 'DFT Oscillator Strength (a.u.)',
    })
    
    new_col_names = []
    for col in df.columns:
        if 'Torsion' in col or 'SD' in col:
            new_col_names.append(col + ' (°)')
        elif 'Polarizability' in col or 'α' in col:
            new_col_names.append(col + ' (Å$^3$)')
        elif 'Dipole' in col:
            new_col_names.append(col + ' (Debye)')
        else:
            new_col_names.append(col)
    
    df.columns = new_col_names
    
    if len(torsion_selection.split('-')[0]) < 2:
        df = df.rename(columns = {
            'Total Torsion Angle_z (°)': 'Total Torsion Angle (°)',
            'Mean Torsion Angle_z (°)': 'Mean Torsion Angle (°)',
            'Inner Mean Torsion Angle_z (°)': 'Inner Mean Torsion Angle (°)',
        })
        
    if len(sd_selection.split('-')[0]) < 2:
        df = df.rename(columns = {
            'SD Torsions_x (°)': 'SD Torsions (°)',
            'SD Difference_x (°)': 'SD Difference (°)'
        })
        
    df = df.dropna().reset_index(drop = True)
    
    print(f'Number of features (m): {len(df.columns)}')
    print(f'Number of data points (n): {len(df)}')
    if df_type == 'gfn':
        print(f"Input features: {list(dict.fromkeys([x.split('_')[0] for x in df.columns]))}\n")
    else:
        print(f"Input features: {list(dict.fromkeys([x.split('_')[0] for x in df.drop(['DFT CD Intensity (a.u.)'], axis = 1).columns]))}\n")
    return df

In [ ]:
def graph_feature_importances(features, importances, fp_sum, fi_method, params, preview):
    def _default_graph(features, importances, add_fp, fp_sum, fi_method, params, preview):
        _, suffix, output_path, palette_list, color_scheme = params
        
        if color_scheme == 'cb10':
            my_cmap = ListedColormap(palette_list, name='cb10')
        else:
            my_cmap = plt.get_cmap(color_scheme)
            
        fi_method_name = '-SHAP' if fi_method == 'SHAP' else ''
        
        importances = np.asarray(importances)
        feature_names = np.asarray(features)
        
        if 'all' in feature_code:
            for i in range(len(feature_names)):
                if '_x (°)' in feature_names[i]:
                    feature_names[i] = feature_names[i].replace('_x (°)', '_0 (°)')
                elif '_y (°)' in feature_names[i]:
                    feature_names[i] = feature_names[i].replace('_y (°)', '_1 (°)')
                elif '_z (°)' in feature_names[i]:
                    feature_names[i] = feature_names[i].replace('_z (°)', '_2 (°)')
    
        indices = np.argsort(importances)
        num_features = min(10, len(importances)) 
        
        fig, ax = plt.subplots(figsize=(10,6))
        
        if add_fp == True:
            custom_colors = ['#291EAB'] + [my_cmap(i / num_features) for i in reversed(range(num_features))]
            graph_filename = f"RF_MIF{fi_method_name}_{suffix}_ecfpsum"
            
            ax.barh(range(num_features +1), np.append(fp_sum, importances[indices[-num_features:]]), color = custom_colors, edgecolor = 'k', align='center')
            plt.yticks(range(num_features +1),  ['ECFP4 Bits Sum'] + [feature_names[i] for i in indices[-num_features:]], fontsize = 16)
            
        else:
            custom_colors = [my_cmap(i / num_features) for i in reversed(range(num_features))]
            graph_filename = f"RF_MIF{fi_method_name}_{suffix}"
            
            ax.barh(range(num_features), importances[indices[-num_features:]], color = custom_colors, edgecolor = 'k', align='center')
            plt.yticks(range(num_features), [feature_names[i] for i in indices[-num_features:]], fontsize = 16)
        
        xlabel_name = 'Mean Absolute SHAP Value' if fi_method == 'SHAP' else 'Relative Importance' 
        plt.xlabel(xlabel_name, fontsize = 20)
        plt.xticks(fontsize = 16)
        plt.tight_layout()
        plt.savefig(os.path.join(output_path, f'{graph_filename}.pdf'))
        
        if preview == False:
            plt.close()
    
    feature_code = params[0]
    if 'y-ecfp' in feature_code:
        for add_fp in [True, False]:
            _default_graph(features, importances, add_fp, fp_sum, fi_method, params, preview)
    else:
        _default_graph(features, importances, False, None, fi_method, params, preview)


def get_feat_imp(data_df, regr, train_X, params, preview):
    def _default(data_df, regr, X, params, preview):
        feature_code, suffix, output_path, _, _ = params
        suffix = params[1]
        output_path = params[2]
        
        print('Default feature importances:')
        for x,y in sorted(zip(data_df.columns, regr.feature_importances_), reverse = True, key = lambda x: x[1])[:10]:
            print(f'{x}: {y*100:.2f}%')
        print()
        
        feature_importances = sorted(zip(data_df.columns, regr.feature_importances_), reverse = True, key = lambda x: x[1])
        feature_df = pd.DataFrame(feature_importances, columns = ['Feature', 'Importance'])
        feature_df['Importance %'] = feature_df['Importance']*100
        feature_df.to_csv(os.path.join(output_path, f"RF_FI_{suffix}.csv"), index = False)

        if 'y-ecfp' in feature_code:
            fp_sum = 0
            for item in sorted(zip(data_df.columns, regr.feature_importances_)):
                if 'ECFP' in item[0]:
                   fp_sum += item[1]
            print(f"\nSum of fingerprint importances: {fp_sum}\n")
        else:
            fp_sum = None
        
        graph_feature_importances(X.columns, regr.feature_importances_, fp_sum, 'default', params, preview)
    
    
    def _shap(regr, train_X, params, preview):
        feature_code, suffix, output_path, _, _ = params
        
        explainer = shap.Explainer(regr, train_X)
        shap_values = explainer(train_X, check_additivity=False)
        
        shap_df = pd.DataFrame(
            {
                'Feature': X.columns,
                'SHAP Value': np.abs(shap_values.values).mean(axis = 0)
            }
        ).sort_values(by = ['SHAP Value'], ascending = False, ignore_index = True)

        print('SHAP Importances')
        for index, row in shap_df.head(10).iterrows():
            print(f"{row['Feature']}: {row['SHAP Value']}")
        print()
        
        shap_df.to_csv(os.path.join(output_path, f"RF_SHAP-FI_{suffix}.csv"), index = False)

        if 'y-ecfp' in feature_code:
            fp_sum = 0
            for index, row in shap_df.iterrows():
                if 'ECFP' in row['Feature']:
                    fp_sum += shap_df.loc[index, 'SHAP Value']
            print(f"Sum of fingerprint importances: {fp_sum}")
        else:
            fp_sum = None

        graph_feature_importances(shap_df['Feature'], shap_df['SHAP Value'], fp_sum, 'SHAP', params, preview)
    
    X = data_df.drop(columns = ['DFT CD Intensity (a.u.)'])
    
    _shap(regr, train_X, params, preview)
    _default(data_df, regr, X, params, preview)

In [ ]:
def graph_predictions(pdf, params, preview):
    def _separate_oligomers(pdf, xlim_max, xlim_max_padded, legend_type, suffix, output_path, preview):
        plt.figure(figsize=(8,6), facecolor='w', edgecolor='k')
        plt.grid(color='lightgray', ls = '-.', lw = 0.1)
        reg_x = np.linspace(0,xlim_max)
        
        pfu_plt = pt_plt = pse_plt = None
        fu_r_value = t_r_value = se_r_value = 0.0
        fu_rmse = t_rmse = se_rmse = np.nan

        if len(pdf[pdf['Oligomer Type'] == 1]) > 0:
            plt.scatter(
                pdf[pdf['Oligomer Type'] == 1]['B97-3c CD Intensity'], 
                pdf[pdf['Oligomer Type'] == 1]['Trained CD Intensity'], 
                marker = 'X', s = 30, c = '#DC267F', edgecolors = 'none', alpha = 0.75, zorder = 1, label = "Furans")
            slope, intercept, fu_r_value, _, _ = linregress(
                pdf[pdf['Oligomer Type'] == 1]['B97-3c CD Intensity'].astype(float),
                pdf[pdf['Oligomer Type'] == 1]['Trained CD Intensity'])
            pfu_plt = plt.plot(reg_x,slope*reg_x + intercept, c = '#DC267F', lw = 2, alpha = 1, zorder = 2)
            fu_rmse = sqrt(mean_squared_error(
                pdf[pdf['Oligomer Type'] == 1]['B97-3c CD Intensity'],
                pdf[pdf['Oligomer Type'] == 1]['Trained CD Intensity']))

        if len(pdf[pdf['Oligomer Type'] == 2]) > 0:
            plt.scatter(
                pdf[pdf['Oligomer Type'] == 2]['B97-3c CD Intensity'], 
                pdf[pdf['Oligomer Type'] == 2]['Trained CD Intensity'], 
                s = 20, c = '#648FFF', edgecolors = 'none', alpha = 0.75, zorder = 0, label = "Thiophenes")
            slope, intercept, t_r_value, _, _ = linregress(
                pdf[pdf['Oligomer Type'] == 2]['B97-3c CD Intensity'].astype(float),
                pdf[pdf['Oligomer Type'] == 2]['Trained CD Intensity'])
            pt_plt = plt.plot(reg_x,slope*reg_x + intercept, c = '#648FFF', lw = 2, alpha = 1, zorder = 2)
            t_rmse = sqrt(mean_squared_error(
                pdf[pdf['Oligomer Type'] == 2]['B97-3c CD Intensity'],
                pdf[pdf['Oligomer Type'] == 2]['Trained CD Intensity']))

        if len(pdf[pdf['Oligomer Type'] == 3]) > 0:
            plt.scatter(
                pdf[pdf['Oligomer Type'] == 3]['B97-3c CD Intensity'], 
                pdf[pdf['Oligomer Type'] == 3]['Trained CD Intensity'], 
                marker = '^', s = 30, c = '#FFB000', edgecolors = 'none', alpha = 0.8, zorder = 1, label = "Selenophenes")
            slope, intercept, se_r_value, _, _ = linregress(
                pdf[pdf['Oligomer Type'] == 3]['B97-3c CD Intensity'].astype(float),
                pdf[pdf['Oligomer Type'] == 3]['Trained CD Intensity'])
            pse_plt = plt.plot(reg_x,slope*reg_x + intercept, c = '#FFB000', lw = 2, alpha = 1, zorder = 2)
            se_rmse = sqrt(mean_squared_error(
                pdf[pdf['Oligomer Type'] == 3]['B97-3c CD Intensity'],
                pdf[pdf['Oligomer Type'] == 3]['Trained CD Intensity']))

        slope, intercept, total_r_value, _, _ = linregress(
            pdf['B97-3c CD Intensity'].astype(float),
            pdf['Trained CD Intensity'])
        total_plt, = plt.plot(reg_x,slope*reg_x + intercept, c = '#785EF0',ls = '--', lw = 2, alpha = 1, zorder = 2, label = 'Total')
        total_rmse = sqrt(mean_squared_error(
            pdf['B97-3c CD Intensity'],
            pdf['Trained CD Intensity']))

        plt.xlabel('B97-3c CD Intensity (a.u.)', fontsize = 18)
        plt.ylabel('Predicted CD Intensity (a.u.)', fontsize = 18)
        plt.xticks(fontsize = 16)
        plt.yticks(fontsize = 16)
        plt.xlim(-10, xlim_max_padded)

        if legend_type == 'separate':
            legend_name = 'sep'
            overall_legend = plt.legend(
                [total_plt],
                ['Total\n$R^2$ = {:.3f}\nRMSE = {:.3f}'.format(total_r_value ** 2, total_rmse)],
                loc='upper left', fontsize=12, frameon=True
            )

            plt.gca().add_artist(overall_legend)

            plt.legend([pfu_plt[0], pt_plt[0], pse_plt[0]], 
                       ['Furans\n$R^2$ = {:.3f}\nRMSE = {:.3f}'.format(fu_r_value ** 2, fu_rmse), 
                        'Thiophenes\n$R^2$ = {:.3f}\nRMSE = {:.3f}'.format(t_r_value ** 2, t_rmse),
                        'Selenophenes\n$R^2$ = {:.3f}\nRMSE = {:.3f}'.format(se_r_value ** 2, se_rmse),],
                       loc = 'lower center', ncol=3, bbox_to_anchor=(0.5, -0.33), markerscale = 1.5, fontsize = 12, frameon = True)
        
            plt.subplots_adjust(bottom=0.18)

        else:
            legend_name = 'bothbot'
            ax = plt.gca()

            fam_handles, fam_labels = [], []
            if 'pfu_plt' in locals() and pfu_plt is not None:
                fam_handles.append(Line2D([0], [0], color='#DC267F', lw=2))
                fam_labels.append('Furans\n$R^2$ = {:.3f}\nRMSE = {:.3f}'.format(fu_r_value**2, fu_rmse))
            if 'pt_plt' in locals() and pt_plt is not None:
                fam_handles.append(Line2D([0], [0], color='#648FFF', lw=2))
                fam_labels.append('Thiophenes\n$R^2$ = {:.3f}\nRMSE = {:.3f}'.format(t_r_value**2, t_rmse))
            if 'pse_plt' in locals() and pse_plt is not None:
                fam_handles.append(Line2D([0], [0], color='#FFB000', lw=2))
                fam_labels.append('Selenophenes\n$R^2$ = {:.3f}\nRMSE = {:.3f}'.format(se_r_value**2, se_rmse))

            if fam_handles: 
                legend_fam = ax.legend(
                    fam_handles, fam_labels,
                    loc='lower center', ncol=3, bbox_to_anchor=(0.5, -0.44),
                    markerscale=1.5, fontsize=12, frameon=True,
                    columnspacing=1.2, handlelength=2.0, borderpad=0.6
                )
                ax.add_artist(legend_fam)

            total_caption = 'Total\n$R^2$ = {:.3f}\nRMSE = {:.3f}'.format(total_r_value**2, total_rmse)
            total_handle = Line2D([0], [0], color='#785EF0', ls='--', lw=2, alpha=1)
            legend_total = ax.legend(
                [total_plt], [total_caption],
                loc='lower center', bbox_to_anchor=(0.5, -0.65),
                frameon=True, fontsize=12, handlelength=2, handletextpad=1.0, borderpad = 0.6
            )
            
            plt.subplots_adjust(bottom=0.30)
        plt.tight_layout()
        plt.savefig(os.path.join(output_path, f'RF_sepolig-TP_{legend_name}_{suffix}.pdf'))
        if preview == False:
            plt.close()
            
    def _combined_oligomers(pdf, xlim_max, xlim_max_padded, suffix, output_path, preview):
        plt.figure(figsize=(8, 6), facecolor='w', edgecolor='k')
        plt.grid(color='lightgray', ls = '-.', lw = 0.1)
        reg_x = np.linspace(0, xlim_max)

        sct = plt.scatter(pdf['B97-3c CD Intensity'], pdf['Trained CD Intensity'], s = 30, c = '#648FFF', edgecolors = 'none', alpha = 0.6)
        slope, intercept, r_value, p_value, std_err = linregress(pdf['B97-3c CD Intensity'].astype(float),pdf['Trained CD Intensity'])
        plt.plot(reg_x,slope*reg_x + intercept, c ='#648FFF', lw = 2, alpha = 1)
        rmse = sqrt(mean_squared_error(pdf['B97-3c CD Intensity'],pdf['Trained CD Intensity']))

        plt.xlabel('B97-3c CD Intensity (a.u.)', fontsize = 18)
        plt.ylabel('Predicted CD Intensity (a.u.)', fontsize = 18)
        plt.xticks(fontsize = 16)
        plt.xlim(-10, xlim_max_padded)
        plt.yticks(fontsize = 16)

        plt.legend(['$R^2$ = {:.3f}\nRMSE = {:.3f}'.format(r_value ** 2, rmse)], 
                   loc = 'upper left', markerscale = 1.5, fontsize = 12, frameon = True)
        
        plt.tight_layout()
        plt.savefig(os.path.join(output_path, f"RF_combolig-TP_{suffix}.pdf"))
        if preview == False:
            plt.close()
    
    suffix = params[1]
    output_path = params[2]
    xlim_max = pdf['B97-3c CD Intensity'].max()
    xlim_max_padded = math.ceil(pdf['B97-3c CD Intensity'].max() / 10) * 10
    
    if 'pt' not in params[0]:
        _separate_oligomers(pdf, xlim_max, xlim_max_padded, 'separate', suffix, output_path, preview)
        _separate_oligomers(pdf, xlim_max, xlim_max_padded, 'combined', suffix, output_path, preview)
        
    _combined_oligomers(pdf, xlim_max, xlim_max_padded, suffix, output_path, preview)



def get_predictions_all(regr, test_X, test_y, params, preview):
    feature_code = params[0]
    suffix = params[1]
    output_path = params[2]
    
    test_y_predictions = np.asarray(regr.predict(test_X))
    pdf = test_X[['Monomer 1','Monomer 2']].copy(deep=True) if feature_code.endswith('_pt') else test_X[['Monomer 1', 'Monomer 2', 'Oligomer Type']].copy(deep = True)
    pdf['B97-3c CD Intensity'] = np.asarray(test_y)
    pdf['Trained CD Intensity'] =  [x for x in list(test_y_predictions)]
    pdf['PctDiff'] = (abs(pdf['B97-3c CD Intensity']-pdf['Trained CD Intensity'])/pdf['B97-3c CD Intensity'])*100
    print('In the test set there are...')
    
    if feature_code.endswith('_pt'):
        print(f"{len(pdf)} molecules")
        predicted_points = pdf[['Monomer 1', 'Monomer 2', 'B97-3c CD Intensity', 'Trained CD Intensity', 'PctDiff']]
    else:
        print(f"Furans: {len(pdf[pdf['Oligomer Type'] == 1])}")
        print(f"Thiophenes: {len(pdf[pdf['Oligomer Type'] == 2])}")
        print(f"Selenophenes: {len(pdf[pdf['Oligomer Type'] == 3])}\n")
        predicted_points = pdf[['Monomer 1', 'Monomer 2', 'Oligomer Type', 'B97-3c CD Intensity', 'Trained CD Intensity', 'PctDiff']]
    
    predicted_points.to_csv(os.path.join(output_path, f"PP_{suffix}.csv"), index = False)
    
    graph_predictions(pdf, params, preview)

In [ ]:
def graph_common_monomer_freq(common_monomers, common_mono_df, params, preview):
    def _normal_monofreq(common_monomers, suffix, output_path, preview):
        plt.figure(figsize=(10, 5), facecolor='w', edgecolor='k')
        n, bins, patches = plt.hist(common_monomers, bins = range(1,367,1))
        fracs = n / n.max()
        norm = colors.Normalize(fracs.min(), fracs.max())
        for thisfrac, thispatch in zip(fracs, patches):
            color = plt.cm.viridis(norm(thisfrac))
            thispatch.set_facecolor(color)

        plt.xlabel('Monomer Number', fontsize = 16)
        plt.ylabel('Frequency', fontsize = 16)
        plt.xticks(fontsize = 16)
        plt.yticks(fontsize = 16)
        plt.xlim(0,367)
        
        plt.title('Monomer Frequency in the Best Performing Hexamers')
        plt.tight_layout()
        plt.savefig(os.path.join(output_path, f"RF_PCM_{suffix}.pdf"))
        if preview == False:
            plt.close()
    
    def _desc_monofreq(common_mono_df, suffix, output_path, preview):
        plt.figure(figsize=(10, 5), facecolor='w', edgecolor='k')
        my_cmap = plt.get_cmap(name = 'viridis_r')
        sorted_df = common_mono_df.sort_values(by = 'Count', ascending = False)
        plt.bar(range(1, 367), sorted_df['Count'].values, color = my_cmap(np.arange(0,125,0.05)))

        plt.ylabel('Frequency', fontsize = 18)
        plt.tick_params(
            axis='x',
            which='both',
            bottom=False,
            top=False,
            labelbottom=False)
        plt.yticks(fontsize = 16)
        plt.xlim(0,367)
        
        plt.title('Monomer Frequency in the Best Performing Hexamers')
        plt.tight_layout()
        plt.savefig(os.path.join(output_path, f"RF_PCMS_{suffix}-300.pdf"))
        if preview == False:
            plt.close()
    
    suffix = params[1]
    output_path = params[2]
    
    _normal_monofreq(common_monomers, suffix, output_path, preview)
    _desc_monofreq(common_mono_df, suffix, output_path, preview)




def predict_all_pts(data_df, num_trees, params, preview):
    feature_code = params[0]
    suffix = params[1]
    output_path = params[2]
    
    data_df = data_df.drop(columns = ['DFT Oscillator Strength (a.u.)'])
    
    X = data_df.drop(['DFT CD Intensity (a.u.)'], axis = 1)
    y = data_df['DFT CD Intensity (a.u.)']
    
    rf_regr = RandomForestRegressor(n_estimators = num_trees, max_features = None, min_samples_split=2, random_state=0, verbose = 0)
    rf_regr.fit(X, y)
    
    all_pts = pd.read_csv('gfn_screening_set.csv.zst', compression = 'zstd').dropna().reset_index(drop = True)
    all_pts = process_df(all_pts, feature_code, 'gfn')
    print(f"all_pts: {list(dict.fromkeys([(x.split('_')[0], all_pts[x].dtype) for x in all_pts.columns]))}\n")
    
    old_pts = data_df.drop(columns = ['DFT CD Intensity (a.u.)']).dropna().reset_index(drop = True)
    print(f"old_pts (input df) length: {len(old_pts)}")
    print(f"all_pts (full df) length: {len(all_pts)}\n")
    all_pts.describe()
    
    new_pts = pd.concat([old_pts, all_pts], ignore_index = True, sort=False).drop_duplicates(subset = ['Monomer 1', 'Monomer 2'], keep = False).dropna(axis = 1)
    print(f"new_pts length: {len(new_pts)}")
    new_pts.describe()
    
    pt_predictions = rf_regr.predict(new_pts)
    new_pts['Trained DFT CD Intensity (a.u.)'] = pt_predictions
    
    common_pts = new_pts.drop(columns = [c for c in new_pts.columns if 'ECFP' in c])
    common_pts = common_pts.sort_values(by = ['Trained DFT CD Intensity (a.u.)'], ascending = False)[:150]
    common_pts.to_csv(os.path.join(output_path, f"RF_CH_{suffix}.csv"), index = False)
    
    high_pts = new_pts[new_pts['Trained DFT CD Intensity (a.u.)'] > 300].sort_values(by = ['Trained DFT CD Intensity (a.u.)'], ascending = False)[['Monomer 1', 'Monomer 2', 'Trained DFT CD Intensity (a.u.)']]
    high_pts.to_csv(os.path.join(output_path, f"RF_HH_{suffix}.csv"), index = False)
    
    common_monomers = list(common_pts['Monomer 1'].astype(int)) + list(common_pts['Monomer 2'].astype(int))
    common_mono_df = pd.DataFrame(columns = ['Monomer', 'Count'])
    rows = []
    
    for i in range(1, 367):
        count = common_monomers.count(i)
        rows.append({'Monomer': i, 'Count': count})
    
    common_mono_df = pd.DataFrame(rows, columns = ['Monomer', 'Count'])
    cm_df = common_mono_df.sort_values(by = ['Count'], ascending = False)
    cm_df.to_csv(os.path.join(output_path, f"RF_CM_{suffix}-300.csv"), index = False)
    graph_common_monomer_freq(common_monomers, common_mono_df, params, preview)
    

In [ ]:
def main(file_name, ttrs, preview):
    color_palette_10 = ['#24E4BC', '#00CFF9', '#648FFF', '#785EF0', '#AA42B8', '#DC267F', '#ED2F2C', '#FE6100', '#FFB000', '#FFE200']
    color_palette_5 = ['#648FFF', '#785EF0', '#DC267F', '#FE6100', '#FFB000']
    input_file = 'clean_dft_df_4096.csv'
    
    num_trees, init_params = set_params(file_name, ttrs)
    params = init_params + [color_palette_10, 'cb10']
    print('Params:')
    for x in range(len(params)):
        print(params[x])
    print()
    
    df = pd.read_csv(input_file)
    if params[0].endswith('_pt'):
        df = df[df['Oligomer Type'] == 2]
        
    data_df = df.copy(deep=True).dropna().reset_index(drop = True)
    data_df = process_df(data_df, params[0], 'dft')
    
    X = data_df.drop(['DFT CD Intensity (a.u.)'], axis = 1)
    y = data_df['DFT CD Intensity (a.u.)']

    train_X, test_X, train_y, test_y = train_test_split(X, y, test_size=0.2, random_state=ttrs)
    n_cols = len(X.columns)
    
    regr = RandomForestRegressor(n_estimators = num_trees, max_features = None, min_samples_split=2, random_state=0, verbose = 0)
    
    start_time = time.time()
    regr.fit(train_X, train_y)
    end_time = time.time()
    
    print(f"Time to train regressor: {end_time - start_time} seconds")
    print(f"Regressor score: {regr.score(test_X, test_y)}\n")
    
    get_feat_imp(data_df, regr, train_X, params, preview)
    get_predictions_all(regr, test_X, test_y, params, preview)
    
    if '_pt' in params[0]:
        predict_all_pts(data_df, num_trees, params, preview)

In [ ]:
filenames = [
    'old-old-y-all-all', 
    'old-old-y-z-x-pt',
    'old-old-y-z-x', 
    'new-old-y-z-x', 
    'new-new-y-z-x', 
]

In [ ]:
ttrs = 42

In [ ]:
for file_name in filenames:
    main(file_name, ttrs, preview = False)